
# PatchTST Inference Notebook (RMSE, all features)

**Goal**: Train a global PatchTST model with RMSE loss using **all numeric features** in `train.csv`, then infer 7-day targets for each provided `TEST_*.csv`.  
**I/O**: Lookback `L=28`, Horizon `H=7`. Output per test window: `./result/submission_TEST_XX.csv` with rows = target dates and columns = `store_menu`.

> Notes  
> - Uses training-only statistics for normalization.  
> - Negative `sales` predictions are clipped to 0.  
> - If a checkpoint exists, it is loaded; otherwise a brief training run executes.  
> - Comments are in concise English.


In [8]:

# Assertions and imports
import os, sys, glob, json, math, time, random, warnings
from dataclasses import dataclass
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# Basic assertions
assert torch.__version__ >= "1.10", "Requires PyTorch >= 1.10"
print("Torch:", torch.__version__)

# Reproducibility
def set_seed(seed:int=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Paths
CANDIDATE_ROOTS = ["./dataset", "/mnt/data/dataset", "/mnt/data"]
def find_first(*relpaths):
    for root in CANDIDATE_ROOTS:
        p = os.path.join(root, *relpaths)
        if os.path.exists(p):
            return p
    return None

TRAIN_PATH = find_first("train.csv") or "/mnt/data/train.csv"
assert TRAIN_PATH and os.path.exists(TRAIN_PATH), f"train.csv not found. Checked: {CANDIDATE_ROOTS}"

TEST_FILES = []
for i in range(10):
    p = find_first(f"TEST_{i:02d}.csv") or f"/mnt/data/TEST_{i:02d}.csv"
    if os.path.exists(p):
        TEST_FILES.append(p)

RESULT_DIR = "./result"
os.makedirs(RESULT_DIR, exist_ok=True)
ART_DIR = "./artifacts_patchtst"
os.makedirs(ART_DIR, exist_ok=True)

# Problem constants
L = 28   # lookback
H = 7    # horizon
PATCH_LEN = 7   # reasonable for L=28
STRIDE = 1
D_MODEL = 128
NHEAD = 8
ENC_LAYERS = 3
FFN_FACTOR = 2
DROPOUT = 0.1
BATCH_SIZE = 256
EPOCHS = 15   # keep moderate; adjust as needed
LR = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
AMP = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# Artifacts
CKPT_PATH = os.path.join(ART_DIR, "patchtst_rmse_allfeatures_best.pt")
CFG_PATH  = os.path.join(ART_DIR, "patchtst_rmse_allfeatures_config.json")
LOG_PATH  = os.path.join(ART_DIR, "training_log_rmse.csv")


Torch: 2.8.0+cu128
Device: cuda


In [9]:

# Metrics
def smape(y_true: np.ndarray, y_pred: np.ndarray, eps: float=1e-6) -> float:
    # Safe sMAPE (in percent)
    num = np.abs(y_pred - y_true)
    den = (np.abs(y_true) + np.abs(y_pred)).clip(eps)
    return 200.0 * np.mean(num / den)

def rmse_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

class RMSELoss(nn.Module):
    """RMSE with epsilon, differentiable."""
    def __init__(self, eps:float=1e-8):
        super().__init__()
        self.eps = eps
    def forward(self, pred, target):
        mse = torch.mean((pred - target)**2)
        return torch.sqrt(mse + self.eps)

# Utilities
def ensure_numeric(df: pd.DataFrame) -> pd.DataFrame:
    """Cast obvious numeric columns; leave non-numeric as-is."""
    for c in df.columns:
        if df[c].dtype == "object":
            try:
                df[c] = pd.to_numeric(df[c])
            except:
                pass
    return df

def find_columns(df: pd.DataFrame) -> Dict[str, str]:
    """Identify key columns present in data."""
    cols = df.columns.tolist()
    must = ["date", "store", "menu", "store_menu", "sales"]
    for m in must:
        assert m in cols, f"Missing required column: {m}"
    return {"date":"date", "store":"store", "menu":"menu",
            "store_menu":"store_menu", "sales":"sales"}

def clip_sales(x: np.ndarray) -> np.ndarray:
    return np.maximum(x, 0.0)


In [10]:

# Scaler: column-wise mean/std computed on training only
class ColumnStandardScaler:
    def __init__(self):
        self.mean_ = None
        self.std_  = None
        self.cols  = None

    def fit(self, X: pd.DataFrame, cols: List[str]):
        self.cols = list(cols)
        m = X[self.cols].astype(float).mean(axis=0).values
        s = X[self.cols].astype(float).std(axis=0).replace(0, 1e-6).values if isinstance(X, pd.DataFrame) else np.std(X, axis=0)
        s = np.where(s == 0, 1e-6, s)
        self.mean_ = m
        self.std_  = s
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        assert self.mean_ is not None and self.std_ is not None
        Z = X.copy()
        Z[self.cols] = (Z[self.cols].astype(float) - self.mean_) / self.std_
        return Z

    def inverse_transform_array(self, arr: np.ndarray, col_idx: int) -> np.ndarray:
        """Inverse only one column by its index within self.cols."""
        mu = self.mean_[col_idx]
        sd = self.std_[col_idx]
        return arr * sd + mu

# Dataset
class GlobalTSWindows(Dataset):
    """Create sliding windows across all store_menu groups, global batches."""
    def __init__(self, df: pd.DataFrame, key_cols: Dict[str,str], feature_cols: List[str],
                 L:int, H:int, scaler: ColumnStandardScaler, mode:str="train"):
        assert mode in {"train", "val"}
        self.k = key_cols
        self.feature_cols = feature_cols
        self.L, self.H = L, H
        self.sales_idx = self.feature_cols.index(self.k["sales"])  # channel index for sales

        self.df = df.sort_values([self.k["store_menu"], self.k["date"]]).reset_index(drop=True)
        # Build index of windows
        self.index = []  # tuples: (start_row_idx_in_group, group_start, group_end, group_id)
        for sm, g in self.df.groupby(self.k["store_menu"]):
            g = g.reset_index(drop=True)
            n = len(g)
            # Available windows
            for start in range(0, n - (L + H) + 1):
                self.index.append((sm, start))

        # Shuffle for train
        if mode == "train":
            random.shuffle(self.index)

        self.scaler = scaler

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx:int):
        sm, start = self.index[idx]
        g = self.df[self.df[self.k["store_menu"]] == sm].reset_index(drop=True)
        sl = slice(start, start + self.L)
        tl = slice(start + self.L, start + self.L + self.H)

        X = g.loc[sl, self.feature_cols].to_numpy(dtype=np.float32)  # (L, C)
        y = g.loc[tl, self.k["sales"]].to_numpy(dtype=np.float32)    # (H,)

        # Clip label to avoid negatives
        y = np.maximum(y, 0.0)

        # Channels-first (C, L)
        X = X.T.astype(np.float32).copy()    # (C, L)
        y = y.astype(np.float32).copy()      # (H,)
        return torch.from_numpy(X), torch.from_numpy(y)



In [11]:

# PatchTST (channel-independent backbone shared across channels)
class PatchEmbed(nn.Module):
    """Convert (B, 1, L) to patch tokens (B, N, D) by unfolding and linear projection."""
    def __init__(self, L:int, patch_len:int, stride:int, d_model:int):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride
        self.proj = nn.Linear(patch_len, d_model)
        # compute N tokens given L
        self.n_patches = 1 + (L - patch_len) // stride
        assert self.n_patches > 0, "Invalid patch config for given L"
        # positional embedding
        self.pos = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)

    def forward(self, x):
        # x: (B, 1, L)
        B, _, L = x.shape
        xs = x.unfold(dimension=2, size=self.patch_len, step=self.stride)  # (B, 1, N, P)
        xs = xs.squeeze(1)  # (B, N, P)
        xs = self.proj(xs)  # (B, N, D)
        xs = xs + self.pos
        return xs  # (B, N, D)

class PatchTST(nn.Module):
    def __init__(self, L:int, H:int, C:int, sales_idx:int, patch_len:int, stride:int,
                 d_model:int, nhead:int, enc_layers:int, dropout:float):
        super().__init__()
        self.C, self.H, self.L = C, H, L
        self.sales_idx = sales_idx

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*2, dropout=dropout, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=enc_layers)
        self.embed = PatchEmbed(L=L, patch_len=patch_len, stride=stride, d_model=d_model)
        self.head = nn.Linear(self.embed.n_patches * d_model, H)

    def forward(self, x):
        # Expect (B, C, L). If (B, L, C), fix by transposing once.
        if x.dim() != 3:
            raise RuntimeError(f"Expected 3D input, got {tuple(x.shape)}")
        if x.shape[1] == self.L and x.shape[2] == self.C:
            x = x.transpose(1, 2)  # (B, L, C) -> (B, C, L)
        assert x.shape[1] == self.C and x.shape[2] == self.L, \
            f"Expected (B,{self.C},{self.L}), got {tuple(x.shape)}"

        B, C, L = x.shape
        x = x.reshape(B*C, 1, L)
        tok = self.embed(x)
        z = self.encoder(tok)
        z = z.reshape(B*C, -1)
        out = self.head(z).reshape(B, C, self.H)
        return out[:, self.sales_idx, :]



In [12]:

# Training loop
@dataclass
class TrainState:
    best_rmse: float = float("inf")
    best_epoch: int = -1

def train_model(train_ds, val_ds, feature_cols, sales_idx, cfg_path: str, ckpt_path:str):
    device = torch.device(DEVICE)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    model = PatchTST(L=L, H=H, C=len(feature_cols), sales_idx=sales_idx,
                     patch_len=PATCH_LEN, stride=STRIDE,
                     d_model=D_MODEL, nhead=NHEAD, enc_layers=ENC_LAYERS, dropout=DROPOUT).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(EPOCHS, 1))
    scaler = torch.cuda.amp.GradScaler(enabled=(AMP and device.type=="cuda"))
    loss_fn = RMSELoss()

    state = TrainState()
    os.makedirs(os.path.dirname(LOG_PATH), exist_ok=True)
    with open(LOG_PATH, "w") as f:
        f.write("epoch,train_rmse,val_rmse\n")

    for epoch in range(1, EPOCHS+1):
        model.train()
        tr_losses = []
        for xb, yb in train_loader:
            # debug once
            print("batch shape:", tuple(xb.shape)); break
            xb = xb.to(device)       # (B, C, L)
            yb = yb.to(device)       # (B, H)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(AMP and device.type=="cuda")):
                pred = model(xb)     # (B, H)
                # Loss on original scale: yb already original; pred currently normalized? No. Inputs normalized only.
                loss = loss_fn(pred, yb)
            scaler.scale(loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update()
            tr_losses.append(loss.item())
        sch.step()
        train_rmse = float(np.mean(tr_losses))

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device); yb = yb.to(device)
                pred = model(xb)
                loss = loss_fn(pred, yb)
                val_losses.append(loss.item())
        val_rmse = float(np.mean(val_losses)) if len(val_losses)>0 else np.nan

        with open(LOG_PATH, "a") as f:
            f.write(f"{epoch},{train_rmse:.6f},{val_rmse:.6f}\n")
        print(f"Epoch {epoch:03d} | train RMSE {train_rmse:.4f} | val RMSE {val_rmse:.4f}")

        # Track best
        if val_rmse < state.best_rmse:
            state.best_rmse = val_rmse
            state.best_epoch = epoch
            torch.save({"model": model.state_dict(),
                        "cfg": {"L":L, "H":H, "C":len(feature_cols),
                                "sales_idx": sales_idx, "patch_len":PATCH_LEN, "stride":STRIDE,
                                "d_model":D_MODEL, "nhead":NHEAD, "enc_layers":ENC_LAYERS,
                                "dropout":DROPOUT}},
                       ckpt_path)

    # Save config
    with open(cfg_path, "w") as f:
        json.dump({"feature_cols": feature_cols, "sales_idx": sales_idx,
                   "L":L, "H":H, "scaler":"column-wise", "patch_len":PATCH_LEN,
                   "stride":STRIDE, "d_model":D_MODEL, "nhead":NHEAD,
                   "enc_layers":ENC_LAYERS, "dropout":DROPOUT}, f, indent=2)

    print("Best epoch:", state.best_epoch, "Best val RMSE:", round(state.best_rmse, 6))
    return ckpt_path


In [13]:

# Load training data
df_train = pd.read_csv(TRAIN_PATH)
df_train = ensure_numeric(df_train)

k = find_columns(df_train)
for c in [k["date"], k["store"], k["menu"], k["store_menu"], k["sales"]]:
    assert c in df_train.columns, f"Missing required column: {c}"

# Strict ordering
df_train[k["date"]] = pd.to_datetime(df_train[k["date"]])
df_train = df_train.sort_values([k["store_menu"], k["date"]]).reset_index(drop=True)

# Feature selection: use ALL numeric columns. Must include 'sales' as one channel.
num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
assert k["sales"] in num_cols, "'sales' must be numeric"
feature_cols = sorted(list(set(num_cols)))
sales_idx = feature_cols.index(k["sales"])

print("Feature columns:", feature_cols)
print("C =", len(feature_cols))

# Clip negative sales in train
df_train[k["sales"]] = df_train[k["sales"]].clip(lower=0)

# Fit scaler on training only
scaler = ColumnStandardScaler().fit(df_train, feature_cols)

# Split train/val by last few windows per group
def split_by_time(df: pd.DataFrame, val_days:int=35):
    parts = []
    parts_val = []
    for sm, g in df.groupby(k["store_menu"]):
        g = g.sort_values(k["date"])
        # last val_days go to validation tail; ensure >= L+H
        if len(g) >= (L+H+val_days):
            train_g = g.iloc[: -val_days].copy()
            val_g   = g.iloc[-(L+H+val_days):].copy()  # let val windows exist
        else:
            # if too short, all to train
            train_g = g.copy()
            val_g   = g.iloc[ max(0, len(g)-(L+H+7)) : ].copy()
        parts.append(train_g); parts_val.append(val_g)
    return pd.concat(parts, ignore_index=True), pd.concat(parts_val, ignore_index=True)

df_tr, df_val = split_by_time(df_train, val_days=H*4)

# Transform numeric features using training stats
df_tr_tf  = scaler.transform(df_tr)
df_val_tf = scaler.transform(df_val)

# Build datasets
train_ds = GlobalTSWindows(df_tr_tf, key_cols=k, feature_cols=feature_cols, L=L, H=H, scaler=scaler, mode="train")
val_ds   = GlobalTSWindows(df_val_tf, key_cols=k, feature_cols=feature_cols, L=L, H=H, scaler=scaler, mode="val")

print("Train windows:", len(train_ds), "Val windows:", len(val_ds))


Feature columns: ['date_ordinal', 'sales']
C = 2
Train windows: 90710 Val windows: 5597


In [14]:

# Train or load
if os.path.exists(CKPT_PATH):
    print("Found checkpoint. Skipping training.")
else:
    print("No checkpoint found. Start training.")
    train_model(train_ds, val_ds, feature_cols, sales_idx, CFG_PATH, CKPT_PATH)


No checkpoint found. Start training.
batch shape: (256, 2, 29)


RuntimeError: stack expects each tensor to be equal size, but got [8] at entry 0 and [7] at entry 28

In [ ]:

# Inference utilities
@torch.no_grad()
def load_model_for_infer(ckpt_path:str, feature_cols:List[str], sales_idx:int):
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    cfg = ckpt["cfg"]
    model = PatchTST(L=cfg["L"], H=cfg["H"], C=cfg["C"], sales_idx=cfg["sales_idx"],
                     patch_len=cfg["patch_len"], stride=cfg["stride"], d_model=cfg["d_model"],
                     nhead=cfg["nhead"], enc_layers=cfg["enc_layers"], dropout=cfg["dropout"]).to(DEVICE)
    model.load_state_dict(ckpt["model"])
    model.eval()
    return model

def infer_on_test_file(test_csv_path:str, model, scaler:ColumnStandardScaler,
                       feature_cols:List[str], sales_idx:int, key_cols:Dict[str,str],
                       L:int, H:int, out_dir:str):
    # Read and basic checks
    df = pd.read_csv(test_csv_path)
    df = ensure_numeric(df)
    for c in [key_cols["date"], key_cols["store_menu"], key_cols["sales"]]:
        assert c in df.columns, f"Missing column in test: {c}"
    df[key_cols["date"]] = pd.to_datetime(df[key_cols["date"]])
    df = df.sort_values([key_cols["store_menu"], key_cols["date"]]).reset_index(drop=True)

    # Transform numeric features using training scaler
    df_tf = df.copy()
    num_in_df = [c for c in feature_cols if c in df_tf.columns]
    missing = [c for c in feature_cols if c not in num_in_df]
    if missing:
        raise ValueError(f"Test file missing required numeric features: {missing}")
    df_tf = scaler.transform(df_tf)

    # Build (B, C, L) for each store_menu using the last 28 days
    X_list, metas = [], []
    target_dates_all = []

    for sm, g in df_tf.groupby(key_cols["store_menu"]):
        g = g.sort_values(key_cols["date"])
        if len(g) < L:
            continue
        # last L rows become model input
        gL = g.iloc[-L:].copy()
        X = gL[feature_cols].to_numpy(dtype=np.float32).T  # (C, L)
        X_list.append(X)
        metas.append(sm)

        last_date = gL[key_cols["date"]].iloc[-1]
        target_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=H, freq="D")
        target_dates_all.append(target_dates)

    if len(X_list) == 0:
        raise ValueError("No groups with at least L rows in test.")

    xb = torch.from_numpy(np.stack(X_list, axis=0)).to(DEVICE)  # (B, C, L)
    with torch.no_grad():
        pred = model(xb)  # (B, H)
    pred = pred.cpu().numpy()
    pred = np.clip(pred, 0.0, None)

    # Build submission-like DataFrame: rows=target dates, cols=store_menu
    # Align by assuming all groups have same target dates window; if not, fill NaN.
    unique_dates = sorted(set().union(*[set(d) for d in target_dates_all]))
    sub = pd.DataFrame(index=pd.to_datetime(unique_dates))
    for sm, p, td in zip(metas, pred, target_dates_all):
        s = pd.Series(p, index=td)
        sub[sm] = s
    sub = sub.sort_index()

    # Save
    base = os.path.basename(test_csv_path).replace(".csv","")
    out_path = os.path.join(out_dir, f"submission_{base}.csv")
    sub.to_csv(out_path, index_label="date")
    print(f"Saved: {out_path}  shape={sub.shape}")
    return out_path


In [ ]:

# Load model
model = load_model_for_infer(CKPT_PATH, feature_cols, sales_idx)

# Infer for each available TEST file
outputs = []
for tf in sorted(TEST_FILES):
    try:
        outp = infer_on_test_file(tf, model, scaler, feature_cols, sales_idx, k, L, H, RESULT_DIR)
        outputs.append(outp)
    except Exception as e:
        print(f"[Skip] {tf}: {e}")

print("Inference complete. Files:", outputs)


In [ ]:

# Combine submissions into a long-form file for convenience
def melt_submission(path:str):
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"])
    out = df.melt(id_vars=["date"], var_name="store_menu", value_name="pred_sales")
    base = os.path.basename(path).replace(".csv","")
    out["submission_id"] = base
    return out

combined = []
for p in sorted(glob.glob(os.path.join(RESULT_DIR, "submission_TEST_*.csv"))):
    try:
        combined.append(melt_submission(p))
    except Exception as e:
        print(p, e)

if combined:
    comb = pd.concat(combined, ignore_index=True)
    long_path = os.path.join(RESULT_DIR, "submission_all_long.csv")
    comb.to_csv(long_path, index=False)
    print("Saved combined long-form:", long_path, comb.shape)
else:
    print("No per-window submissions to combine.")


In [ ]:

print("Done. You can adjust hyperparameters up top and re-run training or only inference.")
print("Artifacts:")
print(" - Checkpoint:", CKPT_PATH, os.path.exists(CKPT_PATH))
print(" - Train log:", LOG_PATH, os.path.exists(LOG_PATH))
for p in sorted(glob.glob(os.path.join(RESULT_DIR, "submission_TEST_*.csv"))):
    print(" -", p)
